# 解答③ RAG パイプライン

> **講師用**: 演習 `ex_03_rag.ipynb` の完全解答です。

In [ ]:
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')

documents = [
    Document(
        page_content='''
RAG（Retrieval-Augmented Generation）は外部ドキュメントを検索して LLM に渡す手法です。
SFT と比べて知識の更新が容易です。チャンキング、エンベッディング、ベクトル検索、生成の4ステップで構成されます。
チャンクサイズが小さいとピンポイントで情報を取得できますが文脈が途切れやすく、
大きいとノイズが増えます。実務では 512〜1024 token が多く使われます。
        ''',
        metadata={'title': 'RAG概要'},
    ),
    Document(
        page_content='''
LangChain は LLM アプリケーション開発のフレームワークです。
ドキュメントローダー、テキストスプリッター、ベクトルストア、LLM チェーンを統合的に扱えます。
ChromaDB は軽量なベクトルデータベースで、ローカル環境でも動作します。
sentence-transformers/all-MiniLM-L6-v2 は日本語にも対応した多言語埋め込みモデルです。
        ''',
        metadata={'title': 'LangChain と ChromaDB'},
    ),
]

In [ ]:
# 解答: RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '。', ' ', ''],
)

chunks = splitter.split_documents(documents)
print(f'チャンク数: {len(chunks)}')

In [ ]:
# 解答: HuggingFaceEmbeddings と Chroma
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print('ベクトルDB作成完了')

In [ ]:
# 解答: similarity_search
query = 'チャンクサイズはどう選べばいいですか？'
results = vectorstore.similarity_search(query, k=3)

print(f'質問: {query}')
for i, doc in enumerate(results):
    print(f'\n[{i+1}] {doc.page_content[:150]}')

In [ ]:
# 実験課題の解答: チャンクサイズ比較
comparison_query = 'ベクトルDBとは何ですか？'

for chunk_size in [256, 512, 1024]:
    sp = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=50)
    ch = sp.split_documents(documents)
    vs = Chroma.from_documents(ch, embeddings, collection_name=f'sol_{chunk_size}')
    docs = vs.similarity_search(comparison_query, k=2)
    print(f'\n=== chunk_size={chunk_size} ({len(ch)} チャンク) ===')
    for d in docs:
        print(f'  {d.page_content[:100].strip()}...')